# CADMIA quick start

CADMIA running in the browser on outputs of an Al₂TiO₅ cluster that ship with the repository
(ORCA 6 and Gaussian 16, PBE0/def2-TZVP). The quantum chemistry was run beforehand; here CADMIA
reads the outputs, analyzes them and runs the steps that need no DFT.

Run the cells in order with **Shift + Enter**.

## 1. Install and check the environment

In [ ]:
![ -d /content/CADMIA ] || git clone -q --depth 1 https://github.com/Padalher/CADMIA.git /content/CADMIA
%cd /content/CADMIA
!pip install -q ".[full]"
!cadmia doctor

`cadmia doctor` lists what is available. ORCA, Gaussian and SLURM are not installed in Colab,
so the calculation commands are unavailable here; every analysis command works. The version
alone:

In [ ]:
!cadmia --version

## 2. Check the outputs

Before any number is computed, `validate` checks each output: normal termination, SCF and
geometry convergence, spin contamination, imaginary frequencies and atom clashes.

In [ ]:
!cadmia validate tests/fixtures/1tial_d4/*.out

## 3. Conceptual DFT

Three single points of the cluster with N, N+1 and N−1 electrons (ORCA 6). CADMIA computes the
global descriptors and the condensed Fukui functions, and ranks the reactive sites.

In [ ]:
!cadmia cdft --n tests/fixtures/1tial_d4/1tial_N.out \
             --np1 tests/fixtures/1tial_d4/1tial_anion.out \
             --nm1 tests/fixtures/1tial_d4/1tial_cation.out \
             --cluster Al2TiO5 -o cdft.json --report cdft_report.txt

The full report, with the per-atom values:

In [ ]:
!cat cdft_report.txt

### Comparison with Multiwfn

The same cluster computed with Gaussian 16, against the report Multiwfn wrote from the same
wavefunctions. CADMIA computes the descriptors on its own; Multiwfn is only the reference.

In [ ]:
from cadmia.cdft.compare import compare_to_multiwfn
from cadmia.cdft.workflow import run_workflow_from_files

F = "tests/fixtures/1tial_cdft/"
result = run_workflow_from_files(F + "N.out", F + "N+1.out", F + "N-1.out")[0]
for row in compare_to_multiwfn(result, F + "CDFT.txt").global_rows:
    print(f"{row.name:8}  CADMIA {row.cadmia:8.4f}   Multiwfn {row.multiwfn:8.4f}   |Δ| {row.abs_diff:.4f}")

Condensed Fukui functions depend on the population scheme. The Gaussian outputs carry Mulliken
charges and Multiwfn uses Hirshfeld, so the per-atom values are compared with the same Hirshfeld
charges:

In [ ]:
from IPython.display import Markdown
Markdown(open("docs/cdft_vs_multiwfn.md").read())

## 4. Structure

Coordination numbers, Hoppe ECN, bonds, coordination polyhedra, rings and point group of the
same cluster, read straight from the ORCA output of the neutral.

In [ ]:
!cadmia structure report tests/fixtures/1tial_d4/1tial_N.out -o /content/structure.txt
!cadmia structure report tests/fixtures/1tial_d4/1tial_N.out -o /content/structure.json
!cat /content/structure.txt

Bond lengths by bond type, as a static figure ready for a paper:

In [ ]:
!cadmia structure plot tests/fixtures/1tial_d4/1tial_N.out --kind bonds -o bonds.png
from IPython.display import Image
Image("bonds.png")

## 5. Adsorption geometries

`geometry` places the probe on the sites ranked by the CDFT step, in every binding mode of the
probe and several orientations. The sampling is kept small here: two sites, two azimuths and two
polar angles (30 configurations). The default, every site with six azimuths and four polar
angles, gives 728 for this cluster.

In [ ]:
!cadmia geometry --cdft cdft.json --xyz tests/fixtures/1tial_d4/1tial_N.out --probe CO2 \
                 --cluster Al2TiO5 --n-sites 2 --n-configs 2 --polar 0 --polar 40 -o co2_configs

## 6. Pre-screening with GFN2-xTB

xTB optimizes every starting geometry, merges those that relax to the same minimum and keeps the
lowest ones for DFT. The official static xTB build runs in Colab; this takes about a minute.

In [ ]:
![ -x /content/xtb-dist/bin/xtb ] || (wget -q https://github.com/grimme-lab/xtb/releases/download/v6.7.1/xtb-6.7.1-linux-x86_64.tar.xz -O /content/xtb.tar.xz && tar -xf /content/xtb.tar.xz -C /content)
!cadmia prescreen co2_configs/*.xyz --xtb-path /content/xtb-dist/bin/xtb -o prescreened

## 7. Adsorption thermochemistry

Optimization and frequency outputs (ORCA 6, PBE0-D4) of the cluster, free CO₂ and one adduct
from an earlier, larger screening of this cluster. CADMIA gives ΔE, ΔH and ΔG of adsorption,
the activation of the probe, and checks whether the cluster rearranged on binding.

In [ ]:
!cadmia analyze tests/fixtures/1tial_d4/1tial_CO2.out \
                --cluster tests/fixtures/1tial_d4/1tial.out \
                --probe-free tests/fixtures/1tial_d4/co2.out \
                --cdft cdft.json -o analysis.json

What the bound CO₂ looks like in an IR spectrum. Free CO₂ has one strong band near
2450 cm⁻¹ (unscaled PBE0); on the cluster it becomes a carbonate, and two new bands appear
near 1560 and 1340 cm⁻¹, where carbonates show up in experimental spectra:

In [ ]:
!cadmia electronic ir tests/fixtures/1tial_d4/co2.out -o ir_co2.png
!cadmia electronic ir tests/fixtures/1tial_d4/1tial_CO2.out -o ir_adduct.png
from IPython.display import Image, display
display(Image("ir_co2.png"), Image("ir_adduct.png"))

## 8. Stability against the pure oxides

Formation energy of the mixed oxide from TiO₂ and Al₂O₃ computed at the same level.

In [ ]:
!cadmia stability --mixed tests/fixtures/1tial_d4/1tial.out \
                  --ref tests/fixtures/1tial_d4/TiO2.out \
                  --ref tests/fixtures/1tial_d4/Al2O3.out -o stability.json

## 9. Report

The CADMIA output: every stage's full report in its own frame, with the level of theory
read from the outputs, the version and the reference to cite. Inside a `cadmia init` project
each stage saves its result to `results/`, and `cadmia report` alone assembles them.

In [ ]:
!cadmia report --structure /content/structure.json --cdft cdft.json --analysis analysis.json \
    --stability stability.json -o cadmia.out
!cat cadmia.out

## What this demo leaves out

CADMIA also prepares and runs the calculations (ORCA or Gaussian, locally or on a SLURM cluster),
chains every step in `cadmia pipeline`, and computes DOS and PDOS, IR spectra, MEP and density
cubes, bond dissociation energies, stability series and descriptor correlations. Every result is
recorded with its provenance in a local database.

To try your own outputs, upload them with the **Files** panel on the left and change the paths
above. `!cadmia --help` lists every command, and `!cadmia guide` prints the user guide.